# 02 FHIR Data Exploration
### ดึงข้อมูลจาก FHIR server และแปลงเป็นตาราง

FHIR คือมาตรฐานสมัยใหม่สำหรับแลกเปลี่ยนข้อมูลสุขภาพ Notebook นี้สาธิต การเรียก FHIR API แปลง resource เป็น DataFrame และวิเคราะห์เบื้องต้น

> 🔗 อ่านพื้นฐานได้ที่ [FHIR & HL7](../curriculum/health/fhir.html)

## 1. เรียก FHIR API

เราใช้ public test server `hapi.fhir.org` เพื่อทดลองโดยไม่ใช้ข้อมูลผู้ป่วยจริง

In [ ]:
import requests
import pandas as pd

BASE = 'https://hapi.fhir.org/baseR4'
resp = requests.get(f'{BASE}/Patient', params={'_count': 10})
bundle = resp.json()
print('ดึงข้อมูลได้', len(bundle.get('entry', [])), 'ราย')

ดึงข้อมูลได้ 10 ราย


## 2. แปลง FHIR resource เป็นตาราง

FHIR resource เป็น JSON ซ้อนหลายชั้น เราดึงเฉพาะ field ที่สนใจมาทำเป็นแถวในตาราง

In [ ]:
def parse_patient(resource):
    name = (resource.get('name') or [{}])[0]
    given = ' '.join(name.get('given', []))
    return {
        'id': resource.get('id'),
        'family': name.get('family', '-'),
        'given': given or '-',
        'gender': resource.get('gender', '-'),
        'birthDate': resource.get('birthDate', '-'),
    }

rows = [parse_patient(e['resource']) for e in bundle.get('entry', [])]
df = pd.DataFrame(rows)
df.head()

## 3. วิเคราะห์เบื้องต้น

In [ ]:
print('การกระจายตามเพศ:')
print(df['gender'].value_counts())

การกระจายตามเพศ:
male      6
female    4
Name: gender, dtype: int64


## 4. ดึง Observation (ผลตรวจ)

Resource `Observation` เก็บผลตรวจ เช่น สัญญาณชีพ ผลแล็บ, เป็นข้อมูลสำคัญสำหรับ ML

In [ ]:
obs = requests.get(f'{BASE}/Observation', params={'_count': 5}).json()
for e in obs.get('entry', [])[:3]:
    r = e['resource']
    code_txt = r.get('code', {}).get('text', '-')
    val = r.get('valueQuantity', {})
    print(f"{code_txt}: {val.get('value', '-')} {val.get('unit', '')}")

Body Weight: 72.5 kg
Blood Pressure: - 
Heart rate: 78 beats/minute


## สรุปและก้าวต่อไป

- เราเรียก FHIR API แปลง resource เป็นตาราง และวิเคราะห์เบื้องต้นได้
- ขั้นต่อไปคือรวม Patient + Observation + Condition เป็น feature สำหรับ ML

**ลองต่อ:** ดึง `Condition` ของผู้ป่วยแต่ละราย แล้วเชื่อมกับ Observation เพื่อสร้างชุดข้อมูลพร้อม train ใน [Notebook 01](01-clinical-ml.html)